In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import time
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType, LongType, DoubleType, StringType



# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

S3_ENDPOINT = "http://minio:9000"
S3_BUCKET   = "s3a://warehouse"

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", S3_BUCKET)
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          S3_ENDPOINT)
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

Spark 4.1.0   catalog: lakehouse


DataFrame[]

In [3]:
# Looking at the taxi trip data
print(os.getcwd())
trips = spark.read.parquet("data/yellow_tripdata_2025-01.parquet")
trips.show(5)
print(trips)

/home/jovyan/project
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|               

In [23]:
# Loading the zone lookup table
print(os.getcwd())
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones.show(5)
print(zones)

/home/jovyan/project
+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows
DataFrame[LocationID: bigint, Borough: string, Zone: string, service_zone: string]


# Bronze layer

In [44]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [45]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.bronze.stg_taxi (
        kafka_time TIMESTAMP,
        key STRING,
        offset INT,
        partition INT,
        value STRING
    ) USING iceberg
""")

parsed_df = raw_stream.select(
    F.col("key").cast("string").alias("key"),
    F.col("value").cast("string").alias("value"),
    F.col("partition").cast("int"),
    F.col("offset").cast("long"),
    F.col("timestamp").alias("kafka_time")
).select("key", "value", "partition", "offset", "kafka_time")

def write_to_iceberg(batch_df, batch_id):
    batch_df = batch_df.withColumn("kafka_time", F.to_timestamp("kafka_time"))

    batch_df.createOrReplaceGlobalTempView("tmp_batch")
    spark.sql("""
        MERGE INTO lakehouse.bronze.stg_taxi t
        USING global_temp.tmp_batch s
            ON t.partition = s.partition AND t.offset = s.offset
        WHEN NOT MATCHED THEN INSERT *
    """)
    # batch-level metrics
    batch_count = batch_df.count()
    if batch_count > 0:
        min_ts = batch_df.agg(F.min("kafka_time")).collect()[0][0]
        max_ts = batch_df.agg(F.max("kafka_time")).collect()[0][0]
    else:
        min_ts = None
        max_ts = None
    
    # log to stdout (visible in driver/executor logs)
    print(f"batch_id={batch_id} count={batch_count} min_kafka_time={min_ts} max_kafka_time={max_ts}")

query = (
    parsed_df.writeStream
      .foreachBatch(write_to_iceberg)
      .option("checkpointLocation", "/tmp/chk-iceberg")
      .trigger(processingTime="5 seconds")
      .start()
)

batch_id=9 count=659905 min_kafka_time=2026-03-28 10:04:25.031000 max_kafka_time=2026-03-28 18:24:13.839000
batch_id=10 count=659 min_kafka_time=2026-03-28 18:24:13.844000 max_kafka_time=2026-03-28 18:24:17.643000
batch_id=11 count=437 min_kafka_time=2026-03-28 18:24:17.649000 max_kafka_time=2026-03-28 18:24:20
batch_id=12 count=935 min_kafka_time=2026-03-28 18:24:20.005000 max_kafka_time=2026-03-28 18:24:24.998000


In [ ]:
# This is here for when you run all cells.
# Gives time to sync some of the data 
time.sleep(10)

In [46]:
# Run this to stop the query
query.stop()

## Testing Bronze layer

In [47]:
# Number of rows
df = spark.sql("""
    SELECT count(*) FROM lakehouse.bronze.stg_taxi
""")
print("Row count")
df.show(1)

# Looking at the data in bronze model
df = spark.sql("""
    SELECT * FROM lakehouse.bronze.stg_taxi
""")
print("Following table shows examples")
df.show(20)

# Checking for duplicates. Should return 0 rows
df = spark.sql("""
    SELECT key, offset, partition, count(*) AS c FROM lakehouse.bronze.stg_taxi
    GROUP BY key, offset, partition
    HAVING c > 1
""")

print("\nFollowing table show duplicates")
df.show(30)

Row count
+--------+
|count(1)|
+--------+
|  679619|
+--------+

Following table shows examples
+--------------------+---+------+---------+--------------------+
|          kafka_time|key|offset|partition|               value|
+--------------------+---+------+---------+--------------------+
|2026-03-28 10:04:...|144|  1865|        0|{"VendorID": 1, "...|
|2026-03-28 10:04:...|158|  1866|        0|{"VendorID": 2, "...|
|2026-03-28 10:04:...|144|  1867|        0|{"VendorID": 1, "...|
|2026-03-28 10:04:...| 48|  1868|        0|{"VendorID": 2, "...|
|2026-03-28 10:04:...| 48|  1869|        0|{"VendorID": 2, "...|
|2026-03-28 10:04:...| 48|  1870|        0|{"VendorID": 2, "...|
|2026-03-28 10:04:...| 48|  1871|        0|{"VendorID": 2, "...|
|2026-03-28 10:04:...| 48|  1872|        0|{"VendorID": 1, "...|
|2026-03-28 10:04:...|158|  1873|        0|{"VendorID": 2, "...|
|2026-03-28 10:04:...|158|  1874|        0|{"VendorID": 2, "...|
|2026-03-28 10:04:...| 48|  1875|        0|{"VendorID": 2,

## Custom scenario

In [49]:
df = spark.sql("""
    SELECT partition, count(*) AS c FROM lakehouse.bronze.stg_taxi
    GROUP BY partition
    ORDER BY c desc
""")

print("\n Showing number of rows per partition")
df.show(30)

df = spark.sql("""
    SELECT key, partition, count(*) FROM lakehouse.bronze.stg_taxi
    group by 1,2 order by 1
""")

print("\n Since all the pickup locations are contained in their partitions, the ordering of taxi trips per pickup location is guaranteed")
df.show(30)


 Showing number of rows per partition
+---------+------+
|partition|     c|
+---------+------+
|        2|200683|
|        3|140471|
|        5|113256|
|        4| 91198|
|        1| 75342|
|        0| 58669|
+---------+------+


 Since all the pickup locations are contained in their partitions, the ordering of taxi trips per pickup location is guaranteed
+---+---------+--------+
|key|partition|count(1)|
+---+---------+--------+
|  1|        3|     161|
| 10|        4|     262|
|100|        3|   10609|
|101|        2|      13|
|102|        3|      13|
|106|        0|      26|
|107|        3|   12842|
|108|        3|      36|
|109|        3|       1|
| 11|        0|      25|
|112|        5|     111|
|113|        4|    8712|
|114|        5|    9185|
|116|        2|     381|
|117|        5|      63|
|119|        3|      60|
| 12|        4|     379|
|120|        1|       5|
|121|        4|      28|
|122|        4|      17|
|123|        5|      48|
|124|        2|      27|
|125|        0| 

# Silver Layer

In [33]:
# SILVER LAYER
taxi_schema = StructType([
    StructField("VendorID", IntegerType()),
    StructField("tpep_pickup_datetime", TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count", IntegerType()),
    StructField("trip_distance", DoubleType()),
    StructField("RatecodeID", IntegerType()),
    StructField("store_and_fwd_flag", StringType()),
    StructField("PULocationID", IntegerType()),
    StructField("DOLocationID", IntegerType()),
    StructField("payment_type", IntegerType()),
    StructField("fare_amount", DoubleType()),
    StructField("extra", DoubleType()),
    StructField("mta_tax", DoubleType()),
    StructField("tip_amount", DoubleType()),
    StructField("tolls_amount", DoubleType()),
    StructField("improvement_surcharge", DoubleType()),
    StructField("total_amount", DoubleType()),
    StructField("congestion_surcharge", DoubleType()),
    StructField("Airport_fee", DoubleType()),
    StructField("cbd_congestion_fee", DoubleType())
])

spark.sql(f"""
    CREATE OR REPLACE TABLE lakehouse.silver.fct_taxi_trip (
        VendorID INT,
        RatecodeID INT,
        PULocationID INT,
        DOLocationID INT,
        tpep_pickup_datetime TIMESTAMP,
        tpep_dropoff_datetime TIMESTAMP,
        passenger_count INT,
        trip_distance DOUBLE,
        store_and_fwd_flag STRING,
        payment_type INT,
        fare_amount DOUBLE,
        extra DOUBLE,
        mta_tax DOUBLE,
        tip_amount DOUBLE,
        tolls_amount DOUBLE,
        improvement_surcharge DOUBLE,
        total_amount DOUBLE,
        congestion_surcharge DOUBLE,
        Airport_fee DOUBLE,
        cbd_congestion_fee DOUBLE,
        PU_Zone STRING,
        PU_Borough STRING,
        PU_service_zone STRING,
        DO_Zone STRING,
        DO_Borough STRING,
        DO_service_zone STRING
    ) USING iceberg
""")

silver_df = (
    spark.table("lakehouse.bronze.stg_taxi")
    .select(F.from_json("value", taxi_schema).alias("d"))
    .select(
        F.col("d.VendorID").cast("int").alias("VendorID"),
        F.col("d.RatecodeID").cast("int").alias("RatecodeID"),
        F.col("d.PULocationID").cast("int").alias("PULocationID"),
        F.col("d.DOLocationID").cast("int").alias("DOLocationID"),
        F.to_timestamp(F.col("d.tpep_pickup_datetime")).alias("tpep_pickup_datetime"),
        F.to_timestamp(F.col("d.tpep_dropoff_datetime")).alias("tpep_dropoff_datetime"),
        F.col("d.passenger_count").cast("int").alias("passenger_count"),
        F.col("d.trip_distance").cast("double").alias("trip_distance"),
        F.col("d.store_and_fwd_flag").cast("string").alias("store_and_fwd_flag"),
        F.col("d.payment_type").cast("int").alias("payment_type"),
        F.col("d.fare_amount").cast("double").alias("fare_amount"),
        F.col("d.extra").cast("double").alias("extra"),
        F.col("d.mta_tax").cast("double").alias("mta_tax"),
        F.col("d.tip_amount").cast("double").alias("tip_amount"),
        F.col("d.tolls_amount").cast("double").alias("tolls_amount"),
        F.col("d.improvement_surcharge").cast("double").alias("improvement_surcharge"),
        F.col("d.total_amount").cast("double").alias("total_amount"),
        F.col("d.congestion_surcharge").cast("double").alias("congestion_surcharge"),
        F.col("d.Airport_fee").cast("double").alias("Airport_fee"),
        F.col("d.cbd_congestion_fee").cast("double").alias("cbd_congestion_fee"),
    )
    .na.fill({
        "passenger_count": 1,
        "trip_distance": 0.0,
        "store_and_fwd_flag": "N",
        "payment_type": 0,
        "fare_amount": 0.0,
        "extra": 0.0,
        "mta_tax": 0.0,
        "tip_amount": 0.0,
        "tolls_amount": 0.0,
        "improvement_surcharge": 0.0,
        "total_amount": 0.0,
        "congestion_surcharge": 0.0,
        "Airport_fee": 0.0,
        "cbd_congestion_fee": 0.0,
        }
    )
)

# joining in the zones
zones = spark.read.parquet("data/taxi_zone_lookup.parquet").select(
    F.col("LocationID").cast("int").alias("LocationID"),
    F.col("Zone"),
    F.col("Borough"),
    F.col("service_zone")
)

z_pu = zones.alias("z_pu")
z_do = zones.alias("z_do")
s = silver_df.alias("s")

silver_enriched = (
    s
    .join(F.broadcast(z_pu), F.col("s.PULocationID") == F.col("z_pu.LocationID"), "left")
    .join(F.broadcast(z_do), F.col("s.DOLocationID") == F.col("z_do.LocationID"), "left")
    .select(
        F.col("s.*"),
        F.col("z_pu.Zone").alias("PU_Zone"),
        F.col("z_pu.Borough").alias("PU_Borough"),
        F.col("z_pu.service_zone").alias("PU_service_zone"),
        F.col("z_do.Zone").alias("DO_Zone"),
        F.col("z_do.Borough").alias("DO_Borough"),
        F.col("z_do.service_zone").alias("DO_service_zone"),
    )
)

silver_enriched.writeTo("lakehouse.silver.fct_taxi_trip").append()

In [34]:
# Number of rows
df = spark.sql("""
    SELECT count(*) FROM lakehouse.silver.fct_taxi_trip
""")
print("Row count")
df.show(1)

# Looking at the data in bronze model
df = spark.sql("""
    SELECT * FROM lakehouse.silver.fct_taxi_trip
""")
print("Following table shows examples")
df.show(10)


Row count
+--------+
|count(1)|
+--------+
|   17683|
+--------+

Following table shows examples
+--------+----------+------------+------------+--------------------+---------------------+---------------+-------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------------+----------+---------------+--------------------+----------+---------------+
|VendorID|RatecodeID|PULocationID|DOLocationID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|             PU_Zone|PU_Borough|PU_service_zone|             DO_Zone|DO_Borough|DO_service_zone|
+--------+----------+------------+------------+--------------------+---------------------+---------------+-------------+-----------

# Gold Layer

The goal is to find the most expensive trips by pickup location. By expensive we mean the most expensive per minute of ride

In [63]:
# Partitioning by months for faster querying, filtering when only looking at a specific time period
# Assuming in a real world case we would have more data than just January and February
# Partitioning by PU_Zone because this field is central to the table and should be used often
spark.sql("""
    CREATE OR REPLACE TABLE lakehouse.gold.analytical_taxi_trips
    USING ICEBERG
    PARTITIONED BY (months(tpep_pickup_datetime), bucket(16, PU_Zone))
    AS
    SELECT
      tpep_pickup_datetime,
      tpep_dropoff_datetime,
      PU_Zone,
      PU_Borough,
      PU_service_zone,
      COALESCE(total_amount,
        fare_amount + extra + mta_tax + tip_amount + tolls_amount + improvement_surcharge + congestion_surcharge + Airport_fee + cbd_congestion_fee
      ) AS total_fee,
      CAST(ROUND((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60.0) AS INT) AS trip_duration_minutes
    FROM lakehouse.silver.fct_taxi_trip
    WHERE tpep_pickup_datetime IS NOT NULL
      AND tpep_dropoff_datetime IS NOT NULL
""")

DataFrame[]

In [66]:
# Just inspecting the data
df = spark.sql("""
    SELECT * FROM lakehouse.gold.analytical_taxi_trips
""")
df.show(10)

# Finding the 10 most expensive PU zones in 2025 January

df = spark.sql("""
    SELECT 
        PU_Zone,
        ROUND(avg(total_fee / NULLIF(trip_duration_minutes, 0)), 2) AS avg_minute_fee,
        count(*) AS number_of_trips,
        ROUND(avg(trip_duration_minutes), 2) AS avg_trip_duration_minutes
    FROM lakehouse.gold.analytical_taxi_trips
    WHERE date_trunc('month', tpep_pickup_datetime) = '2025-01-01' 
    GROUP BY PU_Zone
    ORDER BY avg_minute_fee DESC
""")
df.show(10)

+--------------------+---------------------+-------------------+----------+---------------+---------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|            PU_Zone|PU_Borough|PU_service_zone|total_fee|trip_duration_minutes|
+--------------------+---------------------+-------------------+----------+---------------+---------+---------------------+
| 2025-01-01 00:21:57|  2025-01-01 00:36:23|       Central Park| Manhattan|    Yellow Zone|     19.2|                   14|
| 2025-01-01 00:11:27|  2025-01-01 00:16:58|Little Italy/NoLiTa| Manhattan|    Yellow Zone|     12.2|                    6|
| 2025-01-01 00:24:32|  2025-01-01 00:31:12|       Central Park| Manhattan|    Yellow Zone|    17.16|                    7|
| 2025-01-01 00:28:06|  2025-01-01 00:32:24|Little Italy/NoLiTa| Manhattan|    Yellow Zone|    12.65|                    4|
| 2025-01-01 00:12:23|  2025-01-01 00:19:50|       Central Park| Manhattan|    Yellow Zone|    16.15|                    7|
| 2025-0